# 3장 Cox 모형과 비례위험 점검 (R과 통계분석)

survival 패키지의 원발성 담즙성 간경변 자료(pbc)로 Cox 모형을 맞추고, 비례위험 가정을 점검하고, 보정 생존곡선을 그립니다. 보건통계학 기초의 파이썬 실습 11과 같은 자료입니다. Colab에서는 런타임 → 런타임 유형 변경에서 R을 고른 뒤 셀을 위에서부터 차례로 실행하세요.


## a. 분석할 자료 만들기

### 셀 1. PBC 자료에서 무작위배정 환자 고르기

In [ ]:
library(survival)
d <- subset(pbc, !is.na(trt))      # 무작위배정된 312명만
nrow(d)
table(status = d$status, trt = d$trt)

### 셀 2. 사건, 시간, 공변량 만들기

In [ ]:
d$death  <- as.integer(d$status == 2)   # 사망 = 1, 간이식과 생존 = 0
d$years  <- d$time / 365.25
d$dpca   <- as.integer(d$trt == 1)      # D-penicillamine = 1, 위약 = 0
d$female <- as.integer(d$sex == "f")
d$age10  <- d$age / 10                  # 10세 단위
d$edema  <- factor(d$edema)             # 0, 0.5, 1을 범주로
colSums(is.na(d))[colSums(is.na(d)) > 0]

## b. Cox 모형과 위험비

### 셀 3. 치료만 넣은 Cox 모형

In [ ]:
fit1 <- coxph(Surv(years, death) ~ dpca, data = d)
summary(fit1)
survdiff(Surv(years, death) ~ dpca, data = d)$chisq

### 셀 4. 다변수 Cox 모형

In [ ]:
fit <- coxph(Surv(years, death) ~ dpca + age10 + female + log2(bili) +
               albumin + edema + protime, data = d)
summary(fit)

### 셀 5. 결측이 있는 변수를 넣으면

In [ ]:
coxph(Surv(years, death) ~ dpca + chol, data = d)

### 셀 6. 논문에 옮길 위험비 표

In [ ]:
library(broom)
hr <- tidy(fit, exponentiate = TRUE, conf.int = TRUE)
data.frame(term = hr$term,
           `HR (95% CI)` = sprintf("%.2f (%.2f-%.2f)", hr$estimate,
                                   hr$conf.low, hr$conf.high),
           P = format.pval(hr$p.value, digits = 2, eps = 0.001),
           check.names = FALSE)

### 셀 7. 빌리루빈을 어떤 꼴로 넣을지

In [ ]:
fit_lin <- update(fit, . ~ . - log2(bili) + bili)   # 빌리루빈을 그대로 넣은 모형
AIC(fit_lin, fit)

## c. 비례위험 가정 점검

### 셀 8. cox.zph()로 비례위험 검정

In [ ]:
zph <- cox.zph(fit)
zph

### 셀 9. 시간 변환과 가변수별 검정

In [ ]:
cox.zph(fit, transform = "rank", terms = FALSE)

### 셀 10. 척도화 Schoenfeld 잔차 그림

In [ ]:
par(mfrow = c(1, 2))
plot(zph, var = "dpca")
abline(h = coef(fit)["dpca"], lty = 3, col = "gray40")
plot(zph, var = "protime")
abline(h = coef(fit)["protime"], lty = 3, col = "gray40")

### 셀 11. 부종 범주별 log(-log) 그림

In [ ]:
km_ed <- survfit(Surv(years, death) ~ edema, data = d)
cols <- c("black", "#1f5f8b", "#c8553d")
plot(km_ed, fun = "cloglog", col = cols,
     xlab = "Years (log scale)", ylab = "log(-log S(t))")
legend("topleft", c("No edema", "Edema 0.5", "Edema 1.0"),
       col = cols, lty = 1, bty = "n")

## d. 가정이 어긋날 때와 보정 생존곡선

### 셀 12. 부종으로 층화한 Cox 모형

In [ ]:
fit_s <- coxph(Surv(years, death) ~ dpca + age10 + female + log2(bili) +
                 albumin + protime + strata(edema), data = d)
fit_s

### 셀 13. 추적 기간을 3년에서 나눈 긴 자료

In [ ]:
long <- survSplit(Surv(years, death) ~ ., data = d, cut = 3, episode = "period")
c(nrow(d), nrow(long))
long[long$id %in% c(1, 4), c("id", "tstart", "years", "death", "period", "protime")]

### 셀 14. 기간별 위험비

In [ ]:
long$late <- as.integer(long$period == 2)
fit_t <- coxph(Surv(tstart, years, death) ~ dpca + age10 + female + log2(bili) +
                 albumin + edema + protime + protime:late, data = long)
round(summary(fit_t)$coefficients[c("dpca", "protime", "protime:late"), ], 4)
exp(coef(fit_t)["protime"] + coef(fit_t)["protime:late"])   # 3년 뒤의 위험비

### 셀 15. 직접 표준화한 보정 생존곡선

In [ ]:
std_surv <- function(newd, times)      # 환자마다 예측한 곡선의 평균
  rowMeans(summary(survfit(fit, newdata = newd), times = times)$surv)
at <- c(1, 5, 10)
round(cbind(years   = at,
            dpca    = std_surv(transform(d, dpca = 1), at),
            placebo = std_surv(transform(d, dpca = 0), at)), 3)
grid <- seq(0, 12, by = 0.1)
plot(grid, std_surv(transform(d, dpca = 1), grid), type = "s", ylim = c(0, 1),
     col = "#1f5f8b", xlab = "Years since randomization",
     ylab = "Adjusted survival probability")
lines(grid, std_surv(transform(d, dpca = 0), grid), type = "s", col = "#c8553d")
legend("bottomleft", c("D-penicillamine", "Placebo"),
       col = c("#1f5f8b", "#c8553d"), lty = 1, bty = "n")

## e. 과제 정답

### 셀 16. 과제 1 정답: 빌리루빈과 나이의 위험비

In [ ]:
b <- coef(fit)
exp(b["log2(bili)"] * log2(c(2 / 1, 10 / 5, 11 / 10)))   # 1→2, 5→10, 10→11 mg/dL
exp(2 * b["age10"])                                         # 20세 많을 때

### 셀 17. 과제 2 정답: 간이식도 사건으로 세면

In [ ]:
fit_tx <- update(fit, Surv(years, status > 0) ~ .)    # 사망 또는 간이식
c(death = fit$nevent, death_or_tx = fit_tx$nevent)
round(cbind(death = exp(coef(fit)), death_or_tx = exp(coef(fit_tx))), 3)

### 셀 18. 과제 3 정답: 치료 × 추적 시간

In [ ]:
wrong <- coxph(Surv(years, death) ~ dpca + age10 + female + log2(bili) +
                 albumin + edema + protime + dpca:years, data = d)
round(summary(wrong)$coefficients[c("dpca", "dpca:years"), ], 4)
right <- coxph(Surv(years, death) ~ dpca + age10 + female + log2(bili) +
                 albumin + edema + protime + tt(dpca), data = d,
               tt = function(x, t, ...) x * t)
round(summary(right)$coefficients[c("dpca", "tt(dpca)"), ], 4)